In [1]:
import warnings
from numba.core.errors import NumbaWarning

warnings.simplefilter('ignore', category=NumbaWarning)

import IPython.display as ipd
import torch
from torch.utils.data import DataLoader

import commons
import utils
from data_utils import TextAudioSpeakerLoader, TextAudioSpeakerCollate
from models import SynthesizerTrn
from text.symbols import symbols
from text import text_to_sequence


def get_text(text, hps):
    text_norm = text_to_sequence(text, hps.data.text_cleaners)
    if hps.data.add_blank:
        text_norm = commons.intersperse(text_norm, 0)
    text_norm = torch.LongTensor(text_norm)
    return text_norm

DEBUG:numba.core.byteflow:bytecode dump:
>          0	NOP(arg=None, lineno=1023)
           2	LOAD_FAST(arg=0, lineno=1026)
           4	LOAD_CONST(arg=1, lineno=1026)
           6	BINARY_SUBSCR(arg=None, lineno=1026)
           8	LOAD_FAST(arg=0, lineno=1026)
          10	LOAD_CONST(arg=2, lineno=1026)
          12	BINARY_SUBSCR(arg=None, lineno=1026)
          14	COMPARE_OP(arg=4, lineno=1026)
          16	LOAD_FAST(arg=0, lineno=1026)
          18	LOAD_CONST(arg=1, lineno=1026)
          20	BINARY_SUBSCR(arg=None, lineno=1026)
          22	LOAD_FAST(arg=0, lineno=1026)
          24	LOAD_CONST(arg=3, lineno=1026)
          26	BINARY_SUBSCR(arg=None, lineno=1026)
          28	COMPARE_OP(arg=5, lineno=1026)
          30	BINARY_AND(arg=None, lineno=1026)
          32	RETURN_VALUE(arg=None, lineno=1026)
DEBUG:numba.core.byteflow:pending: deque([State(pc_initial=0 nstack_initial=0)])
DEBUG:numba.core.byteflow:stack: []
DEBUG:numba.core.byteflow:state.pc_initial: State(pc_initial=0 nstack_

/mnt/Shared-Storage/yash/miniconda3/envs/SpeechExpts/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEBUG:graphviz._tools:deprecate positional args: graphviz.backend.piping.pipe(['renderer', 'formatter', 'neato_no_op', 'quiet'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.backend.rendering.render(['renderer', 'formatter', 'neato_no_op', 'quiet'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.backend.unflattening.unflatten(['stagger', 'fanout', 'chain', 'encoding'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.backend.viewing.view(['quiet'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.quoting.quote(['is_html_string', 'is_valid_id', 'dot_keywords', 'endswith_odd_number_of_backslashes', 'escape_unescaped_quotes'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.quoting.a_list(['kwargs', 'attributes'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.quoting.attr_list(['kwargs', 'attributes'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.dot.Dot.clear(['keep_attrs'])
DEBUG:graphviz._tools:deprecate posit

In [2]:
from speech_reconstruction import get_model

model = get_model().cuda()
model.eval()
# _ = utils.load_checkpoint('G_150000.pth', model, None)

Using cache found in /home/yash/.cache/torch/hub/bshall_hubert_main
Downloading: "https://github.com/bshall/hubert/releases/download/v0.2/hubert-soft-35d9f29f.pt" to /home/yash/.cache/torch/hub/checkpoints/hubert-soft-35d9f29f.pt
 37%|███▋      | 132M/361M [00:17<00:29, 8.01MB/s] 


KeyboardInterrupt: 

In [3]:
from speech_reconstruction import CustomDataset, DataCollate

dataset = CustomDataset()
collater = DataCollate()
dataLoader = DataLoader(dataset, collate_fn=collater, num_workers=1, shuffle=False, batch_size=1, pin_memory=True, drop_last=False)
data_list = list(dataLoader)

Using cache found in /home/yash/.cache/torch/hub/bshall_hubert_main
Downloading: "https://github.com/bshall/hubert/releases/download/v0.2/hubert-soft-35d9f29f.pt" to /home/yash/.cache/torch/hub/checkpoints/hubert-soft-35d9f29f.pt
  1%|          | 4.38M/361M [00:04<06:23, 976kB/s] 


KeyboardInterrupt: 

In [4]:
print(len(dataset))

4198


In [8]:
with torch.no_grad():
    y, y_lengths, sid_src = [x.cuda() for x in data_list[0]]
    # print(spec.shape, spec_lengths)
    sid_tgt1 = torch.LongTensor([38]).cuda()
    sid_tgt2 = torch.LongTensor([53]).cuda()
    sid_tgt3 = torch.LongTensor([54]).cuda()
    audio1 = model.infer(y, y_lengths, sid=sid_tgt1)[0,0].data.cpu().float().numpy()
    audio2 = model.infer(y, y_lengths, sid=sid_tgt2)[0,0].data.cpu().float().numpy()
    audio3 = model.infer(y, y_lengths, sid=sid_tgt3)[0,0].data.cpu().float().numpy()
    audio4 = model.infer(y, y_lengths, sid=sid_src)[0,0].data.cpu().float().numpy()
    print(y.shape, audio1.shape, audio2.shape, audio3.shape)    
# print(len(audio1), len(audio1[0]), len(audio1[0][0]))
print("Original SID: %d" % sid_src.item())
ipd.display(ipd.Audio(y[0].cpu().numpy(), rate=24_000, normalize=True))
print("Converted SID: %d" % sid_tgt1.item())
ipd.display(ipd.Audio(audio1, rate=24_000, normalize=True))
print("Converted SID: %d" % sid_tgt2.item())
ipd.display(ipd.Audio(audio2, rate=24_000, normalize=True))
print("Converted SID: %d" % sid_tgt3.item())
ipd.display(ipd.Audio(audio3, rate=24_000, normalize=True))
print("Converted SID: %d" % sid_src.item())
ipd.display(ipd.Audio(audio4, rate=24_000, normalize=True))

TypeError: Model.infer() got multiple values for argument 'sid'